# 05 GAFF2 Parameterisation

This notebook focuses only on **Stage 1: GAFF2 parameterisation with AmberTools**.

In simple terms, parameterisation converts a molecule file such as SDF into files that describe the physical model used for molecular dynamics. The most important outputs for the next notebook are:

- `.prmtop`: AMBER topology and force-field parameters
- `.inpcrd`: AMBER starting coordinates

This v2 workflow keeps the same general idea as iPHASimulator v1: prepare parameters first, then run simulation separately. The implementation here is deliberately simpler and specific to the v2 RDKit-built PHA oligomers.

## What AmberTools does here

The helper function `parameterize_gaff2` runs three AmberTools steps:

1. `antechamber`: assigns GAFF2 atom types and partial charges
2. `parmchk2`: looks for missing force-field terms and writes an `.frcmod` file
3. `tleap`: assembles AMBER `.prmtop` and `.inpcrd` files

For careful production work, the charge model matters. The default is AM1-BCC (`charge_method="bcc"`), which is more appropriate but can be slow. For quick debugging, `charge_method="gas"` avoids AM1-BCC and runs faster.

## Step 1: Check AmberTools

This cell does not run parameterisation. It only checks whether `antechamber`, `parmchk2`, and `tleap` are available on your `PATH`.

In [1]:
from iphasimulator.parameterization.gaff2 import ambertools_available

ambertools_available()

True

If the result is `False`, install the MD dependencies in your conda environment before running GAFF2:

```bash
conda install -c conda-forge ambertools openmm parmed mdtraj -y
```

Then restart Jupyter from the same environment.

## Step 2: Choose the SDF input

Run notebook `04_export_structures.ipynb` first if these SDF files do not exist. The default target below is `PHB4_R` because it is small and fast compared with longer-side-chain systems.

In [3]:
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
output_root = repo_root / "examples" / "output"
structure_root = output_root / "polymer_structures"

target_name = "PHB4_R"
sdf_path = structure_root / f"{target_name}.sdf"
gaff2_output_dir = output_root / "md_tests" / target_name.replace("_R", "") / "gaff2"

{
    "sdf_path": sdf_path,
    "sdf_exists": sdf_path.exists(),
    "gaff2_output_dir": gaff2_output_dir,
}

{'sdf_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/PHB4_R.sdf'),
 'sdf_exists': True,
 'gaff2_output_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2')}

## Step 3: Run GAFF2 parameterisation when ready

The next cell is deliberately disabled by default. Set `RUN_GAFF2 = True` only after:

- `sdf_exists` is `True`
- `ambertools_available()` is `True`
- you are comfortable with the run taking some time

For first debugging, keep `charge_method = "gas"`. For more realistic charge assignment, use `charge_method = "bcc"`.

In [ ]:
from iphasimulator.parameterization.gaff2 import parameterize_gaff2

RUN_GAFF2 = True
charge_method = "abcg2"  # use "abcg2" or "bcc" for AM1-BCC charges

if RUN_GAFF2:
    gaff2_outputs = parameterize_gaff2(
        sdf_path,
        gaff2_output_dir,
        name=target_name.replace("_R", ""),
        net_charge=0,
        residue_name="PHA",
        charge_method=charge_method,
        verbose=True,
    )
    gaff2_outputs
else:
    "Set RUN_GAFF2 = True to run AmberTools parameterisation."

[GAFF2:antechamber] Running: antechamber -i /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/PHB4.antechamber.sdf -fi sdf -o PHB4.gaff2.mol2 -fo mol2 -at gaff2 -c abcg2 -nc 0 -rn PHA -s 2
[GAFF2:antechamber] Completed in 87.1s
[GAFF2:parmchk2] Running: parmchk2 -i PHB4.gaff2.mol2 -f mol2 -o PHB4.gaff2.frcmod -s gaff2
[GAFF2:parmchk2] Completed in 0.1s
[GAFF2:tleap] Running: tleap -f tleap.in
[GAFF2:tleap] Completed in 0.3s


## Step 4: Check expected outputs

After a successful run, `prmtop_exists` and `inpcrd_exists` should be `True`. These two files are the inputs for notebook `06_openmm_setup.ipynb`.

In [5]:
amber_name = target_name.replace("_R", "")
prmtop_path = gaff2_output_dir / f"{amber_name}.prmtop"
inpcrd_path = gaff2_output_dir / f"{amber_name}.inpcrd"

{
    "prmtop_path": prmtop_path,
    "prmtop_exists": prmtop_path.exists(),
    "inpcrd_path": inpcrd_path,
    "inpcrd_exists": inpcrd_path.exists(),
    "timing_log": gaff2_output_dir / "timing.log",
    "antechamber_log": gaff2_output_dir / "antechamber.log",
}

{'prmtop_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/PHB4.prmtop'),
 'prmtop_exists': True,
 'inpcrd_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/PHB4.inpcrd'),
 'inpcrd_exists': True,
 'timing_log': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/timing.log'),
 'antechamber_log': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/antechamber.log')}

## Command-line equivalent

For repeatable runs, use the script from the terminal:

```bash
PYTHONPATH=src python examples/run_gaff2_openmm_test.py --target PHB4 --skip-openmm --skip-am1-bcc
```

Remove `--skip-am1-bcc` when you want AM1-BCC charge generation.

## Roadmap After GAFF2 Parameterisation

GAFF2 parameterisation is the point where the workflow branches. Up to this notebook, the goal has been to build a chemically reasonable PHA oligomer and generate AMBER-style force-field files: `prmtop`, `inpcrd`, `mol2`, and `frcmod`. After this point, the same parameterised polymer can be tested in different molecular simulation engines and environments.

The 06-stage notebooks are deliberately split by engine and system environment:

| Notebook | Responsibility | Use it when |
| --- | --- | --- |
| `06A_openmm_dry_polymer.ipynb` | Quick OpenMM validation of the GAFF2 AMBER files without explicit solvent | you want a fast local smoke test that the parameterised polymer can minimise and run short dynamics |
| `06B_gromacs_dry_polymer.ipynb` | Convert AMBER files to GROMACS inputs and validate a dry polymer GROMACS folder | you need GROMACS `.gro`, `.top`, `.ndx`, `.mdp`, and scripts as the base for later solvated workflows |
| `06C_gromacs_solvated_system.ipynb` | Build the explicit-solvent GROMACS polymer system and generate minimisation/equilibration/production scripts | you want the default solvated polymer workflow for benchmarking and production trajectory generation |
| `06D_openmm_solvated_system.ipynb` | Template for OpenMM-native explicit-solvent setup | you want to explore an OpenMM solvation route once force-field template support for the polymer is available |

### OpenMM vs GROMACS

OpenMM is useful for Python-native testing, local debugging, and direct integration with Python analysis workflows. In this project, the dry OpenMM route is especially useful immediately after GAFF2 because it can quickly catch broken coordinates, bad parameters, or obvious instability before a larger GROMACS workflow is prepared.

GROMACS is the main route for explicit-solvent production-style polymer simulations here. It gives a standard file layout, mature command-line tooling, efficient trajectory output, and HPC-friendly scripts for NVT, NPT, and production runs. Use GROMACS when you need a reproducible production trajectory for preprocessing and analysis.

### Dry vs solvated systems

Dry polymer simulations are useful for validation, debugging, and rapid iteration. They answer narrow questions: do the coordinates load, does the topology match the coordinates, can the molecule minimise, and do very short dynamics run without immediate failure? Dry simulations are not a substitute for realistic solution behavior.

Solvated systems are required when the scientific question depends on polymer conformation in water, ion effects, density relaxation, diffusion, solvent exposure, radius of gyration in solution, SASA, or any production trajectory intended for physical interpretation. Explicit solvent also introduces periodic boundary conditions, which is why trajectory preprocessing becomes necessary before RMSD, Rg, or SASA analysis.

### Simplified polymer workflow

The simplified GROMACS polymer workflow is the default for polymer benchmarking and rapid iteration:

| Stage | Purpose | Typical length |
| --- | --- | ---: |
| `step6.0_minimization` | remove bad contacts in the solvated/ionised box | not time-based |
| `step6.1_nvt` | bring the system to 300 K at constant volume | 100 ps |
| `step6.2_npt` | relax pressure and solvent-box density at 1 bar | 500 ps |
| `step7_production` | generate the analysis trajectory | 100 ns by default |

This workflow is short, readable, and easy to debug. It is appropriate for polymer-only boxes, small PHA oligomer benchmarks, method development, and rapid checks before longer production runs. Its main disadvantage is that it is less conservative for heterogeneous systems with delicate interfaces.

### CHARMM-GUI-style staged equilibration workflow

The project also keeps a CHARMM-GUI-style staged GROMACS template in the `gromacs/charmm_gui_membrane/` workflow folder. This workflow is not the default polymer benchmarking path. It is an advanced template for membrane proteins, enzyme/polymer systems, protein/polymer complexes, aggregation systems, and other sensitive heterogeneous systems.

The staged workflow uses more equilibration stages:

| Stage | Purpose | Typical length |
| --- | --- | ---: |
| `step6.0_minimization` | remove steric clashes before dynamics | not time-based |
| `step6.1_equilibration` | initial NVT thermalisation | 50 ps |
| `step6.2_equilibration` | early gentle NPT relaxation | 50 ps |
| `step6.3_equilibration` | continued NPT relaxation | 100 ps |
| `step6.4_equilibration` | further density/interface relaxation | 100 ps |
| `step6.5_equilibration` | production-like NPT relaxation | 200 ps |
| `step6.6_equilibration` | final pre-production NPT check | 200 ps |
| `step7_production` | generate the analysis trajectory | 100 ns by default |

Staged equilibration exists because complex systems can be damaged by abrupt relaxation. Membranes need time to relax area and thickness. Protein complexes and enzyme/polymer interfaces can distort if pressure coupling or solvent rearrangement is too aggressive. Aggregates and mixed systems may need gradual density relaxation before production-like dynamics are safe.

The advantage of the staged approach is scientific caution: temperature, pressure, solvent, and interfaces are relaxed gradually. The disadvantage is cost and complexity: there are more files, more stages to inspect, and longer wall-clock time before production.

### Choosing a path

For fast parameter validation, start with `06A`. For GROMACS production preparation, go through `06B` and `06C`. For polymer-only solvated benchmarking, use the simplified polymer workflow. For membranes, enzyme/polymer systems, protein complexes, or sensitive interfaces, treat the CHARMM-GUI-style staged workflow as the safer advanced template and inspect each equilibration stage before production.
